# A Transformer encoder for text classification

*Corrected version (Chapter 19). The original is kept as `transformers_original.ipynb`.*

The data of the original (BBC news, `df_file.csv` from Kaggle) is used when the file is present. Otherwise the notebook builds a 5-topic subset of the NLTK Reuters corpus, so that it runs anywhere.

In [ ]:
import os, re, math, random
from collections import Counter
import numpy as np
import pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## Data

In [ ]:
if os.path.exists('df_file.csv'):
    df = pd.read_csv('df_file.csv')                  # columns Text, Label (BBC news)
    texts, labels = df['Text'].tolist(), df['Label'].values
else:
    import nltk
    nltk.download('reuters', quiet=True)
    from nltk.corpus import reuters
    TOPICS = ['acq', 'crude', 'trade', 'money-fx', 'interest']
    texts, labels, per = [], [], Counter()
    for f in reuters.fileids():
        c = reuters.categories(f)
        if len(c) == 1 and c[0] in TOPICS and per[c[0]] < 300:
            per[c[0]] += 1; texts.append(reuters.raw(f)); labels.append(TOPICS.index(c[0]))
    labels = np.array(labels)
print(len(texts), 'documents;', Counter(labels.tolist()))
NUM_CLASSES = len(set(labels.tolist()))

## Split first, then build the vocabulary

The original built the vocabulary from all documents and used `list(set(...))`, whose order changes from run to run. Here the split comes first, the vocabulary is built from the **training** documents only and is sorted by frequency (then alphabetically), so the run is reproducible. Index 0 is padding, 1 is the unknown word.

In [ ]:
tok = lambda t: re.findall(r"[a-z]+", t.lower())
idx = np.arange(len(texts))
tr, te = train_test_split(idx, test_size=0.2, random_state=SEED, stratify=labels)

cnt = Counter(w for i in tr for w in tok(texts[i]))
words = [w for w, c in sorted(cnt.items(), key=lambda x: (-x[1], x[0])) if c >= 2][:5000]
word2idx = {'<PAD>': 0, '<UNK>': 1, **{w: i + 2 for i, w in enumerate(words)}}
MAX_LEN = 128

def encode(t):
    ids = [word2idx.get(w, 1) for w in tok(t)][:MAX_LEN]
    return ids + [0] * (MAX_LEN - len(ids))

Xtr = torch.tensor([encode(texts[i]) for i in tr]); ytr = torch.tensor(labels[tr])
Xte = torch.tensor([encode(texts[i]) for i in te]); yte = torch.tensor(labels[te])
print('vocabulary:', len(word2idx), ' train/test:', len(Xtr), len(Xte))

## A fair baseline: TF-IDF + logistic regression

In the original the last cell re-ran the scikit-learn models on `X_train`, which by then held the *padded index sequences* of the Transformer instead of the one-hot vectors; the token IDs are not features, and the accuracy fell to 0.31. Here every model gets its own variables.

In [ ]:
vec = TfidfVectorizer(stop_words='english', min_df=2)
A_tr = vec.fit_transform([texts[i] for i in tr]); A_te = vec.transform([texts[i] for i in te])
lr = LogisticRegression(max_iter=1000).fit(A_tr, labels[tr])
print('TF-IDF + LR accuracy:', round(lr.score(A_te, labels[te]), 4))

## The Transformer encoder

Three changes against the original: (1) a **padding mask**, so that real tokens do not attend to `<PAD>`; (2) **masked mean pooling** over the real tokens instead of the first token, which in the original was just the first word of the text (no `[CLS]` token was ever added); (3) the feed-forward width is $4d$ (it was $d$).

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.h, self.dh = num_heads, d_model // num_heads
        self.qkv = nn.Linear(d_model, 3 * d_model)
        self.out = nn.Linear(d_model, d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):               # pad_mask: (B, L), True = padding
        B, L, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q, k, v = [t.view(B, L, self.h, self.dh).transpose(1, 2) for t in (q, k, v)]
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dh)
        if pad_mask is not None:
            scores = scores.masked_fill(pad_mask[:, None, None, :], float('-inf'))
        a = self.drop(torch.softmax(scores, dim=-1))
        return self.out((a @ v).transpose(1, 2).reshape(B, L, D))

class Block(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.att = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Dropout(dropout), nn.Linear(d_ff, d_model))
        self.n1, self.n2 = nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, pad_mask=None):               # post-LN, as in the original paper
        x = self.n1(x + self.drop(self.att(x, pad_mask)))
        return self.n2(x + self.drop(self.ff(x)))

class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, max_len, num_classes, d_model=64, num_heads=4, num_layers=1):
        super().__init__()
        self.tok = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos = nn.Embedding(max_len, d_model)
        self.blocks = nn.ModuleList([Block(d_model, num_heads, 4 * d_model) for _ in range(num_layers)])
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, x):
        pad = x == 0
        h = self.tok(x) + self.pos(torch.arange(x.shape[1], device=x.device))[None]
        for b in self.blocks:
            h = b(h, pad)
        m = (~pad).float().unsqueeze(-1)
        return self.head((h * m).sum(1) / m.sum(1))

model = TransformerClassifier(len(word2idx), MAX_LEN, NUM_CLASSES)
print('parameters:', sum(p.numel() for p in model.parameters()))

## Training and evaluation

In [ ]:
def pick_device():
    if torch.cuda.is_available():
        try:
            torch.zeros(1, device='cuda'); return torch.device('cuda')
        except Exception:
            pass
    if getattr(torch.backends, 'mps', None) and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

device = pick_device(); print('device:', device)
model.to(device)
opt = torch.optim.Adam(model.parameters(), lr=2e-3)
losses = []
for epoch in range(15):
    model.train(); perm = torch.randperm(len(Xtr)); tot = 0
    for b in range(0, len(Xtr), 32):
        ix = perm[b:b + 32]
        loss = F.cross_entropy(model(Xtr[ix].to(device)), ytr[ix].to(device))
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item() * len(ix)
    losses.append(tot / len(Xtr))
    print(f'epoch {epoch + 1:2d}  loss {losses[-1]:.4f}')

model.eval()
with torch.no_grad():
    pred = model(Xte.to(device)).argmax(1).cpu()
print('Transformer test accuracy:', round((pred == yte).float().mean().item(), 4))
print(classification_report(yte.numpy(), pred.numpy()))